## The following file is for creating file commands.csv, which includes commands for taking the raw hungate files in the folder organization system as downloaded directly from the hungate 1000 database online, found [here](https://genome.jgi.doe.gov/portal/TheHunmicrobiome/TheHunmicrobiome.info.html).
The raw files do not contain ncbi taxonomic IDs, so the commands use seqfu to rename them by pulling the taxonomic ids from taxonomy.csv. This is saved in commands.csv.

In [1]:
import pandas as pd

In [2]:
taxonomic = pd.read_csv("taxonomy.csv")
# Manually shifting things around as needed
taxonomic.iloc[350,2] = taxonomic.iloc[350,3]
taxonomic.iloc[350,3] = taxonomic.iloc[350,4]
taxonomic.iloc[350,4] = None
taxonomic = taxonomic.drop("Unnamed: 4", axis = 1)


files = pd.read_csv("file_structure.csv", header = None)
files = files.drop([0,2,4,5,6,7,8], axis = 1)
files.rename(columns = {1:"whole_path", 3:"name"},inplace = True)

In [3]:
taxonomic

,genus,species,strain,dsm,taxid
0,Ace,rum,DSM,5522,1120918
1,Aci,fer,pGA-4,NaN,905
2,Aci,fer,WCC6,NaN,905
3,Aci,sp,DSM,11652,346222
4,Act,den,PA,NaN,52767
...,...,...,...,...,...
405,Ter,hib,KPPR-9,NaN,2813371
406,Tre,bry,B25,NaN,163
407,Tre,bry,NK4A124,NaN,877418
408,Tre,bry,XBD1002,NaN,163


In [4]:
files

,whole_path,name
0,AcerumDSM5522_FD/Assembly/QC_and_Genome_Assem...,AcerumDSM5522
1,AciferpGA4_FD/Assembly/QC_and_Genome_Assembly...,AciferpGA4
2,AciferWCC6_FD/Assembly/QC_and_Genome_Assembly...,AciferWCC6
3,AcispDSM11652_FD/Assembly/QC_and_Genome_Assem...,AcispDSM11652
4,ActdenPA_FD/Assembly/QC_and_Genome_Assembly/s...,ActdenPA
...,...,...
409,SucrumDSM9236_FD/Assembly/QC_and_Genome_Assem...,SucrumDSM9236
410,TrebryB25_FD/Assembly/QC_and_Genome_Assembly/...,TrebryB25
411,TrebryNK4A124_FD/Assembly/QC_and_Genome_Assem...,TrebryNK4A124
412,TrebryXBD1002_FD/Assembly/QC_and_Genome_Assem...,TrebryXBD1002


Putting the taxonomic information into the same name format as the files:

In [5]:
taxonomic["name"] = taxonomic["genus"] + taxonomic["species"]+ taxonomic["strain"]
taxonomic["name"] += taxonomic["dsm"].where(taxonomic["dsm"].notna(), '').astype(str)
taxonomic["name"] = taxonomic["name"].str.replace(r'[^A-Za-z0-9]', '', regex=True)

In [6]:
import re

def clean_string(s):
    return re.sub(r'\W+', '', str(s)).strip()

taxonomic["name_clean"] = taxonomic["name"].apply(clean_string)
files["name_clean"] = files["name"].apply(clean_string)


In [7]:
combined = pd.merge(taxonomic, files, on="name_clean", how="inner")
combined

,genus,species,strain,dsm,taxid,name_x,name_clean,whole_path,name_y
0,Ace,rum,DSM,5522,1120918,AcerumDSM5522,AcerumDSM5522,AcerumDSM5522_FD/Assembly/QC_and_Genome_Assem...,AcerumDSM5522
1,Aci,fer,pGA-4,NaN,905,AciferpGA4,AciferpGA4,AciferpGA4_FD/Assembly/QC_and_Genome_Assembly...,AciferpGA4
2,Aci,fer,WCC6,NaN,905,AciferWCC6,AciferWCC6,AciferWCC6_FD/Assembly/QC_and_Genome_Assembly...,AciferWCC6
3,Aci,sp,DSM,11652,346222,AcispDSM11652,AcispDSM11652,AcispDSM11652_FD/Assembly/QC_and_Genome_Assem...,AcispDSM11652
4,Act,den,PA,NaN,52767,ActdenPA,ActdenPA,ActdenPA_FD/Assembly/QC_and_Genome_Assembly/s...,ActdenPA
...,...,...,...,...,...,...,...,...,...
377,Suc,dex,ACV-10,NaN,83771,SucdexACV10,SucdexACV10,SucdexACV10_FD/Assembly/QC_and_Genome_Assembl...,SucdexACV10
378,Suc,dex,H5,NaN,1410676,SucdexH5,SucdexH5,SucdexH5_FD/Assembly/QC_and_Genome_Assembly/s...,SucdexH5
379,Tre,bry,B25,NaN,163,TrebryB25,TrebryB25,TrebryB25_FD/Assembly/QC_and_Genome_Assembly/...,TrebryB25
380,Tre,bry,NK4A124,NaN,877418,TrebryNK4A124,TrebryNK4A124,TrebryNK4A124_FD/Assembly/QC_and_Genome_Assem...,TrebryNK4A124


Grabbing just the necessary columns:

In [8]:
mapping = combined[["whole_path","taxid","name_clean"]]

In [ ]:
#saving the mapped info:
mapping.to_csv("mapped.csv", index = False)

In [ ]:
taxonomic.iloc[43,6]

In [9]:
# Manually adjusting taxonomic to add in ones that were not automatically mapped
taxonomic.iloc[43,6] = "BlautiaspSF50"
#taxonomic.iloc[102,6] = "CloamiKH1P1"
taxonomic.iloc[124,6] = "DoreaspAGR2135"
taxonomic.iloc[135,6] = "EubpyruKHGC13"
taxonomic.iloc[220,6] = "Laclacslactis511"
taxonomic.iloc[284,6] = "PrevotellaspRM4"
taxonomic.iloc[312,6] = "RumamyRM87"
taxonomic.iloc[331,6] = "RumflaYRD2003"
taxonomic.iloc[331,6] = "Selbov8141"
taxonomic.iloc[350,6] = "SelrumlacDSM2872"
#new_row = {'taxid': 1526, 'name_clean': 6}
#df.loc[len(df)] = new_row_loc



combined = pd.merge(taxonomic, files, on="name_clean", how="right")


In [10]:
combined[combined["taxid"].isna()] # too see all the ones that didn't make it

,genus,species,strain,dsm,taxid,name_x,name_clean,whole_path,name_y
18,NaN,NaN,NaN,NaN,NaN,NaN,BacphaphiBrb01,BacphaphiBrb01_FD/Assembly/QC_and_Genome_Asse...,BacphaphiBrb01
19,NaN,NaN,NaN,NaN,NaN,NaN,BacphaphiBrb02,BacphaphiBrb02_FD/Assembly/QC_and_Genome_Asse...,BacphaphiBrb02
101,NaN,NaN,NaN,NaN,NaN,NaN,CloamiKH1P1,CloamiKH1P1_FD/Assembly/QC_and_Genome_Assembl...,CloamiKH1P1
112,NaN,NaN,NaN,NaN,NaN,NaN,CloglycoKPPR9,CloglycoKPPR9_FD/Assembly/QC_and_Genome_Assem...,CloglycoKPPR9
221,NaN,NaN,NaN,NaN,NaN,NaN,LacrumWC1T17,LacrumWC1T17_FD/Assembly/QC_and_Genome_Assemb...,LacrumWC1T17
292,NaN,NaN,NaN,NaN,NaN,NaN,ProspMB3007,ProspMB3007_FD/Assembly/QC_and_Genome_Assembl...,ProspMB3007
325,NaN,NaN,NaN,NaN,NaN,NaN,RumbroYE282,RumbroYE282_FD/Assembly/QC_and_Genome_Assembl...,RumbroYE282
327,NaN,NaN,NaN,NaN,NaN,NaN,RumflaMC2020,RumflaMC2020_FD/Assembly/QC_and_Genome_Assemb...,RumflaMC2020
330,NaN,NaN,NaN,NaN,NaN,NaN,RumflaXPD3002,RumflaXPD3002_FD/Assembly/QC_and_Genome_Assem...,RumflaXPD3002
332,NaN,NaN,NaN,NaN,NaN,NaN,RumflaYAD2003,RumflaYAD2003_FD/Assembly/QC_and_Genome_Assem...,RumflaYAD2003


Adding the commands to run:

In [11]:
mapping.loc[:, "command"] = mapping.apply(
    lambda row: f'seqfu cat --append "|kraken:taxid|{row["taxid"]}" {row["whole_path"]} > N_{row["name_clean"]}.fasta',
    axis=1
)

#Properly modified files when command is run will have a N_ prefix to differentiate them

/tmp/ipykernel_1172922/1949254250.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mapping.loc[:, "command"] = mapping.apply(


In [12]:
mapping

,whole_path,taxid,name_clean,command
0,AcerumDSM5522_FD/Assembly/QC_and_Genome_Assem...,1120918,AcerumDSM5522,"seqfu cat --append ""|kraken:taxid|1120918"" Ac..."
1,AciferpGA4_FD/Assembly/QC_and_Genome_Assembly...,905,AciferpGA4,"seqfu cat --append ""|kraken:taxid|905"" Acifer..."
2,AciferWCC6_FD/Assembly/QC_and_Genome_Assembly...,905,AciferWCC6,"seqfu cat --append ""|kraken:taxid|905"" Acifer..."
3,AcispDSM11652_FD/Assembly/QC_and_Genome_Assem...,346222,AcispDSM11652,"seqfu cat --append ""|kraken:taxid|346222"" Aci..."
4,ActdenPA_FD/Assembly/QC_and_Genome_Assembly/s...,52767,ActdenPA,"seqfu cat --append ""|kraken:taxid|52767"" Actd..."
...,...,...,...,...
377,SucdexACV10_FD/Assembly/QC_and_Genome_Assembl...,83771,SucdexACV10,"seqfu cat --append ""|kraken:taxid|83771"" Sucd..."
378,SucdexH5_FD/Assembly/QC_and_Genome_Assembly/s...,1410676,SucdexH5,"seqfu cat --append ""|kraken:taxid|1410676"" Su..."
379,TrebryB25_FD/Assembly/QC_and_Genome_Assembly/...,163,TrebryB25,"seqfu cat --append ""|kraken:taxid|163"" Trebry..."
380,TrebryNK4A124_FD/Assembly/QC_and_Genome_Assem...,877418,TrebryNK4A124,"seqfu cat --append ""|kraken:taxid|877418"" Tre..."


In [ ]:
mapping.to_csv("commands.csv", index = False)

## Post database creation processing below

In [ ]:
custom_counts = pd.read_csv("../data/S_counts.csv")
standard_counts = pd.read_csv("../standard_run_data/S_counts.csv")

In [ ]:
standard_taxa = standard_counts[['genus', 'species']].drop_duplicates()
custom_taxa = custom_counts[['genus', 'species']].drop_duplicates()

merged = standard_taxa.merge(custom_taxa, on=['genus', 'species'], how='left', indicator=True)
not_in_custom = merged[merged['_merge'] == 'left_only']
print("Taxa in standard but NOT in custom:")
print(len(not_in_custom))

merged2 = custom_taxa.merge(standard_taxa, on=['genus', 'species'], how='left', indicator=True)
not_in_standard = merged2[merged2['_merge'] == 'left_only']
print("Taxa in custom but NOT in standard:")
print(len(not_in_standard))


In [ ]:
not_in_standard